# 第34课 · 高频的伪装术 — Nyquist 定理与混叠（aliasing）：6kHz 如何冒充 2kHz

**今日目标**：理解采样率 sr 必须 **> 2×最高频率**（奈奎斯特）。亲眼看到高频正弦被欠采样后「伪装」成低频。

> **闭环**：L02 给过「每个周期至少 2 点」；L04 用 sr=200 和弦翻过车。本课**先预测再运行**：6 kHz 如何冒充 2 kHz。

**为什么对 Aurora 重要**：16 kHz 管道只「信」0–8 kHz；抗混叠滤波器截掉 8 kHz 以上，避免假低频污染 Mel / MFCC。

← **上一课**　[L33 · 正弦波生成](L33_sine_wave.ipynb)

> 上节课学习了 **正弦波生成**：x[n]=A·sin(2πfn/sr)，亲手实现并对齐 aurora.audio.sine。  
> 本课将探讨 **Nyquist 定理与混叠**。

## 本课剧情：CD 音频为什么选 44100 Hz？

人耳能听到的最高频率约 20 kHz。CD 的采样率是 44100 Hz——刚好是 20000 的两倍多一点。

这不是巧合，而是**奈奎斯特定理（Nyquist theorem）**：要不失真地采样频率为 `f` 的信号，采样率必须满足 `sr > 2f`。否则，高频会"折叠"成一个假的低频——这就是**混叠（aliasing）**。

**直觉**：用 8 kHz 采 6 kHz 正弦波，会发生什么？

采样点之间间隔 `1/8000` 秒。6 kHz 波在每个间隔内转了 `6000/8000 = 0.75` 圈——比 0.5 圈多，但数字化系统"看不出" 0.75 圈和 -0.25 圈的区别（都产生同样的采样点序列）。因此 6 kHz 被误认为 `|6000 - 8000·round(6000/8000)| = 2000 Hz`。

### 为什么 0.75 圈和 -0.25 圈产生同样的采样点？

这涉及正弦函数的周期性。在角度上：
- 0.75 圈 = 0.75 × 2π = 1.5π 弧度
- -0.25 圈 = -0.25 × 2π = -0.5π 弧度

关键的三角恒等式是：**sin(θ) = sin(θ - 2π)**（正弦函数周期为 2π）

因此：
$$\sin(1.5\pi) = \sin(1.5\pi - 2\pi) = \sin(-0.5\pi)$$

更直观的说法：**sin(1.5π) = -1**，**sin(-0.5π) = -1**，都等于 -1。

在采样中，当波转了 0.75 圈（1.5π）时的采样点值，与转了 -0.25 圈（-0.5π）时的值完全相同。系统只能看到这个数值，分不出是哪个角度产生的，所以把 0.75 圈的 6 kHz 误认为是 -0.25 圈的某个低频。通过折叠公式计算，这个低频就是 2 kHz。

---

折叠公式：

$$\text{alias}(f, f_s) = \left| f - f_s \cdot \text{round}\!\left(\frac{f}{f_s}\right) \right|$$

本节任务：实现 `predict_alias_freq(freq, sample_rate)` 并用 8 kHz/6 kHz 案例验证。

## 1. 概念

采样率 `sr` 能如实表示的最高频率是 `sr/2`（**Nyquist 频率**）。超过它，高频会折叠成一个假的低频——这就是**混叠**。

### 为什么最高频是 sr/2？

想象一个时钟，你每 1 秒看一次。
- 如果秒针每次转 0.5 圈（转 180°），你看 10 次会看到：0°、180°、0°、180°…… 你能清楚地分辨出它在"快速振动"。
- 如果秒针每次转 0.4 圈（转 144°），你看 10 次会看到：0°、144°、288°、72°…… 完全看不清方向。
- 临界情况：**每次看时秒针恰好转 0.5 圈**。这是分辨的极限。

在采样里：
- 采样间隔 = 1/sr 秒
- 频率为 f 的波在每个采样间隔内转 `f / sr` 圈
- 要分辨这个波，需要 `f / sr` ≤ 0.5，即 **f ≤ sr/2**
- 如果 f > sr/2，相邻采样点之间波转了超过 0.5 圈，系统"看不清"真正的频率，会产生混叠。

**例**：8 kHz 采样率 → Nyquist = 4 kHz。4 kHz 的波在每个采样点之间刚好转 0.5 圈（180°），这是能分辨的极限。

---

### 为什么会折叠

采样只记录离散时刻的振幅，丢失了两次采样之间的连续细节。超过 Nyquist 的频率 `f` 与其镜像频率在每个采样点上产生完全相同的数值，因此两者在离散域无法区分。

**折叠机制**：`round(f/sr)` 找到最近的整数倍采样率（也就是最近的折叠中心 `k·sr`），用来把高频"拉回"到 Nyquist 范围内。`abs(f - k·sr)` 计算真实频率到这个折叠中心的距离，得到视在频率。

**例**：`sr=8000, f=6000` 
- `round(6000/8000) = round(0.75) = 1`，折叠中心是 `1×8000 = 8000` Hz
- 距离 `|6000 - 8000| = 2000 Hz`，所以 6 kHz 听起来像 2 kHz

## 实验入口：把声音拆成可观察的数组

用 `sr=8000` 和极短时长（0.01 s）生成采样点，让数组足够小，可以直接打印。后续代码会对比 `true_freq=6000` 的高频波和折叠后的低频波，验证两者在离散采样点上逐点吻合（正弦向下折叠附带相位翻转，所以 6 kHz 的采样点等于 2 kHz 纯音的反相版本）。

### 复习：sine() 函数长什么样？

接下来会反复调用 `sine(freq, duration, sample_rate)` 生成波形——这是 L33 学过的函数。如果隔了一段时间有点想不起来也没关系，先在这里对齐一下：

`sine(freq, duration, sample_rate)` 实现的就是这个公式：

$$x[n] = \sin\!\left(\frac{2\pi f n}{sr}\right), \quad n = 0, 1, 2, \dots, N-1$$

也就是说，它返回一个数组，数组的第 `n` 个元素是"频率为 `f` 的正弦波在第 `n` 个采样时刻的振幅"。`N`（数组长度）由 `duration × sample_rate` 决定。

这一点很关键：本课接下来要对比"6 kHz 版本"和"2 kHz 版本"两条采样点序列是否完全相同——如果不确定 `sine()` 到底在算什么，就没法判断"逐点吻合"意味着什么。下面代码格会先跑一个小例子，把这个公式变成看得见的数字。

In [1]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [2]:
import numpy as np
from aurora.audio import sine

# 复习 L33：sine(freq, duration, sample_rate) 返回 x[n] = sin(2*pi*freq*n/sample_rate)
# 用一个慢一点、看得清楚的例子（1Hz，用1000Hz"采样率"只是为了在这里打印方便）
demo = sine(freq=1, duration=0.005, sample_rate=1000)
n_demo = np.arange(len(demo))
print('n（第几个采样点）        =', n_demo)
print('sine(1, 0.005, 1000)[n] =', np.round(demo, 3))
print('对照公式：x[n] = sin(2π·1·n/1000)，逐点算出来的值应该和上面一致')
print('\n就绪')

n（第几个采样点）        = [0 1 2 3 4]
sine(1, 0.005, 1000)[n] = [0.    0.006 0.013 0.019 0.025]
对照公式：x[n] = sin(2π·1·n/1000)，逐点算出来的值应该和上面一致

就绪


## 动手观察：序列怎样一步步变成信号

修改 `sample_rate`、`duration`、`freq`，观察打印出的时间轴长度（`N`）和采样点间距（`1/sr`）如何随之变化。

In [3]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('采样点编号 n =', n)
print('时间轴 t =', np.round(t, 3))
print('角度 angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


N = 8
采样点编号 n = [0 1 2 3 4 5 6 7]
时间轴 t = [0.    0.125 0.25  0.375 0.5   0.625 0.75  0.875]
角度 angle = [ 0.     1.571  3.142  4.712  6.283  7.854  9.425 10.996]
sin(angle) = [ 0.  1.  0. -1. -0.  1.  0. -1.]


## 代码实验：把所有频率都折回 Nyquist 范围

这个实验遍历 0–20 Hz 的频率，用 `|f - sr·round(f/sr)|` 计算每个频率在 `sr=10` 时的视在频率，展示 Nyquist 两侧的频率如何对称折叠。

In [4]:
import numpy as np

sample_rate = 10
nyquist = sample_rate / 2
freqs = np.arange(0, 21)

for f in freqs:
    alias = abs(((f + nyquist) % sample_rate) - nyquist)
    print(f'true={f:2d}Hz -> alias={alias:4.1f}Hz')


true= 0Hz -> alias= 0.0Hz
true= 1Hz -> alias= 1.0Hz
true= 2Hz -> alias= 2.0Hz
true= 3Hz -> alias= 3.0Hz
true= 4Hz -> alias= 4.0Hz
true= 5Hz -> alias= 5.0Hz
true= 6Hz -> alias= 4.0Hz
true= 7Hz -> alias= 3.0Hz
true= 8Hz -> alias= 2.0Hz
true= 9Hz -> alias= 1.0Hz
true=10Hz -> alias= 0.0Hz
true=11Hz -> alias= 1.0Hz
true=12Hz -> alias= 2.0Hz
true=13Hz -> alias= 3.0Hz
true=14Hz -> alias= 4.0Hz
true=15Hz -> alias= 5.0Hz
true=16Hz -> alias= 4.0Hz
true=17Hz -> alias= 3.0Hz
true=18Hz -> alias= 2.0Hz
true=19Hz -> alias= 1.0Hz
true=20Hz -> alias= 0.0Hz


## 2. ✏️ 实现混叠频率预测 `predict_alias_freq`

**手算验证（sr=8000）**：

| 真实频率 f | round(f/sr) | f - k·sr | alias |
|---|---|---|---|
| 0 | 0 | 0 | **0 Hz** |
| 2000 | 0 | 2000 | **2000 Hz**（低于 Nyquist，不折叠） |
| 4000 | round(0.5)=0 | 4000 | **4000 Hz**（恰好 Nyquist） |
| 6000 | 1 | -2000 | **2000 Hz**（折叠！） |
| 8000 | 1 | 0 | **0 Hz**（正好一圈，混叠为 DC） |
| 10000 | 1 | 2000 | **2000 Hz** |

规律：频率关于 Nyquist（4000 Hz）镜像折叠，每隔 `sr` 重复一次。

> **实现提示**：`abs(freq - sample_rate * round(freq / sample_rate))`  
> 注意 Python `round(0.5)=0`（银行家舍入），与 NumPy `np.round(0.5)=0.0` 一致。

### round() 函数的含义：寻找"折叠中心"

折叠公式中的 `round(f/sr)` 做的是什么？

**直观理解**：频率 f 最接近哪个"sr 的整数倍"？
- `f/sr` 告诉你 f 是 sr 的多少倍：0.75 倍、1.2 倍、2.3 倍……
- `round(f/sr)` 把这个倍数四舍五入到最近的整数 k
- 这个整数 k 就代表"折叠的第几圈"

**具体例**（sr=8000）：
- f=6000 → 6000/8000=0.75 → round(0.75)=1 → 第 1 圈（折叠中心 1×8000=8000 Hz）
- f=3000 → 3000/8000=0.375 → round(0.375)=0 → 第 0 圈（折叠中心 0×8000=0 Hz）
- f=9000 → 9000/8000=1.125 → round(1.125)=1 → 第 1 圈（折叠中心 1×8000=8000 Hz）
- f=10000 → 10000/8000=1.25 → round(1.25)=1 → 第 1 圈（折叠中心 1×8000=8000 Hz）

一旦确定了折叠中心 k×sr，就用 `|f - k×sr|` 计算 f 到这个中心的距离，这个距离就是"伪装后的假频率"。

### 关于 Python 的 round() 和银行家舍入

Python 用的是"银行家舍入"（banker's rounding）：
- 0.75 → 1 ✓（符合直觉）
- 0.5 → 0（舍到偶数）
- 1.5 → 2（舍到偶数）
- 2.5 → 2（舍到偶数）

在本课的例子里（6000/8000=0.75），根本用不到 0.5 这个边界，所以银行家舍入不会造成问题。但如果你用 NumPy 或其他库，要确认舍入规则是否一致。对于这课的算法，**用 Python 的 round() 和 NumPy 的 np.round() 结果是相同的**。

---

### 看看折叠的具体计算

用 sr=8000 时的几个例子：

### 从"转了几圈"到"折成几赫兹"——补上完整的推导链条

前面已经知道 6000 Hz 被 8000 Hz 采样后会变成 2000 Hz，也知道 `round(0.75)=1` 能算出折叠中心是 8000 Hz。但这中间还差一步：**从"0.75 圈"这个中间量，到"2000 Hz"这个最终答案，到底是怎么一步步走过去的？** 把这条链条完整地摆开：

1. **算比例**：6000 Hz 相当于 `8000 × 0.75`，也就是说，波在每个采样间隔里转了 0.75 圈（这正是 `f/sr = 6000/8000 = 0.75` 的含义）。
2. **发现问题**：0.75 圈已经超过半圈（0.5 圈）这条"能分辨"的红线。系统只能看采样点上的数值，而 `sin(0.75 圈的角度) = sin(-0.25 圈的角度)`（前面用周期性恒等式证过），所以系统没法分辨"正着转了 0.75 圈"和"反着转了 0.25 圈"这两种情况。
3. **换算成频率**：既然 0.75 圈在系统眼里等价于 -0.25 圈，就把这个"圈数"换算回赫兹：`8000 × (-0.25) = -2000` Hz。
4. **去掉方向**：负号只表示"反着转"，人耳或频谱图只关心转动的快慢（大小），不关心方向，所以取绝对值：`|-2000| = 2000` Hz。

链条完整写出来就是：

$$6000\text{ Hz} \;\to\; \frac{6000}{8000}=0.75\text{ 圈} \;\to\; \text{折回} -0.25\text{ 圈} \;\to\; 8000\times(-0.25)=-2000\text{ Hz} \;\to\; |-2000|=2000\text{ Hz}$$

这条链条和 `round()` 公式是同一件事的两种讲法：`round(6000/8000)=1` 只是快速定位"折叠中心"（8000 Hz），而上面这四步说明了中心两侧的 0.75 圈和 -0.25 圈为什么真的是同一件事——这就是"为什么高频会冒充低频"的完整因果链。

### 写代码前，先把变量表补完整

写 `predict_alias_freq` 前明确三件事：
- 输入：`freq`（待测频率，Hz）和 `sample_rate`（采样率，Hz）
- 关键步骤：计算 `k = round(freq / sample_rate)`，再求 `abs(freq - k * sample_rate)`
- 返回：float，即该频率的视在频率（范围 0 到 `sample_rate / 2`）

In [5]:
def predict_alias_freq(freq, sample_rate):
    """返回 freq 被 sample_rate 采样后的视在频率（0 到 Nyquist 范围）。
    折叠公式：alias = abs(freq - sample_rate * round(freq / sample_rate))
    """
    # ✏️ TODO: 实现折叠公式
    raise NotImplementedError(
        "predict_alias_freq: 请实现折叠公式 abs(freq - sr * round(freq/sr))"
    )


In [6]:
# 边界条件验证 — 实现 predict_alias_freq 后运行此格
# 这三个边界恰好是最容易弄错符号或取整方向的地方
try:
    assert predict_alias_freq(0, 8000) == 0.0,    "freq=0 应返回 0.0"
    assert predict_alias_freq(4000, 8000) == 4000.0, "freq=sr/2 应返回 Nyquist 4000.0"
    assert predict_alias_freq(8000, 8000) == 0.0,  "freq=sr 应折回 0.0"
    print('✅ 边界条件全部通过')
except (NotImplementedError, TypeError):
    print('⚠️  请先实现 predict_alias_freq，再运行此格')


⚠️  请先实现 predict_alias_freq，再运行此格


In [ ]:
sr = 8000          # Nyquist = 4000 Hz
true_freq = 6000   # 高于 Nyquist
alias = predict_alias_freq(true_freq, sr)
print('预测视在频率:', alias, 'Hz')
assert 0 <= alias <= sr/2, '视在频率应落在 0..Nyquist'

x_aliased = sine(true_freq, duration=0.01, sample_rate=sr)
x_lowfreq = sine(alias,     duration=0.01, sample_rate=sr)

# 先看看具体的采样点数值
print('\n--- 前 10 个采样点的具体值 ---')
print(f'n    |  6kHz采样点   |  2kHz采样点   | -2kHz采样点  |')
for n in range(min(10, len(x_aliased))):
    print(f'{n:2d}   |  {x_aliased[n]:8.5f}    |  {x_lowfreq[n]:8.5f}    | {-x_lowfreq[n]:8.5f}   |')

# 正弦向下折叠会附带相位翻转：sin(2π(fs−f)n/fs) = −sin(2πfn/fs)
# 所以 6 kHz 的采样点等于 2 kHz 纯音「取负」——同一个 2 kHz 频率，相位差 π。
diff_plus  = float(np.max(np.abs(x_aliased - x_lowfreq)))   # 与 +2kHz 比
diff_minus = float(np.max(np.abs(x_aliased + x_lowfreq)))   # 与 −2kHz 比
print(f'\n与 +2kHz 最大差异: {diff_plus:.2e}')
print(f'与 −2kHz 最大差异: {diff_minus:.2e}')
print('→ 6 kHz 采样点 = 2 kHz 纯音的反相版本（频率相同，相位差 π）')
assert diff_minus < 1e-10, f'混叠验证失败：应与 −2kHz 逐点吻合，得到 {diff_minus:.2e}'
print('✅ 混叠验证通过：离散域无法区分 6 kHz 与 2 kHz 的频率')

## 3. 验证混叠：为什么高频采样点 = 低频反相？

### 相位翻转公式的推导

代码里有一个关键观察：当 6 kHz 被 8 kHz 采样时，采样点序列恰好等于 2 kHz 低频**取负**：

$$\sin\left(2\pi(f_s - f) \frac{n}{f_s}\right) = -\sin\left(2\pi f \frac{n}{f_s}\right)$$

为什么会自动出现负号？

**推导**（用三角恒等式 sin(A - B) = sin A cos B - cos A sin B）：

$$\sin\left(2\pi(f_s - f) \frac{n}{f_s}\right) = \sin\left(2\pi n - 2\pi f \frac{n}{f_s}\right)$$

利用 sin(A - B) 展开：

$$= \sin(2\pi n) \cos\left(2\pi f \frac{n}{f_s}\right) - \cos(2\pi n) \sin\left(2\pi f \frac{n}{f_s}\right)$$

因为 n 是整数：
- $\sin(2\pi n) = 0$（任何整数）
- $\cos(2\pi n) = 1$（任何整数）

所以：

$$= 0 \cdot \cos\left(2\pi f \frac{n}{f_s}\right) - 1 \cdot \sin\left(2\pi f \frac{n}{f_s}\right) = -\sin\left(2\pi f \frac{n}{f_s}\right)$$

**结论**：向下折叠的高频正弦波自动变成了低频的反相版本（相位差 π）。

---

### 用单位圆理解

想象单位圆上两个人在走：
- 红人（6 kHz）：每个采样点转 0.75 圈（270°）
- 蓝人（2 kHz）：每个采样点转 0.25 圈（90°）

0.75 圈和 0.25 圈在圆上的对应位置关于某个轴对称（相位差 180° = π）。所以采样点上，红人的位置总是蓝人位置的反向。

---

## 4. 画图：高频采样点 vs 那个假低频（反相镜像）

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(x_aliased, 'o-', ms=4, label=f'sampled {true_freq} Hz @ {sr} Hz')
plt.plot(x_lowfreq, 'x--', ms=5, label=f'pure {alias:.0f} Hz')
plt.title('Aliasing: a 6 kHz tone masquerades as a low tone')
plt.xlabel('sample n'); plt.legend(); plt.tight_layout(); plt.show()

**结论**：欠采样让高频伪装成低频。这就是录音前要"抗混叠滤波"的原因。

**🎉 完成后**：`git commit -m 'learn: L34 aliasing'`

In [ ]:
# 彩蛋：手动计算几个采样点，验证 0.75 圈 = -0.25 圈
import numpy as np

sr = 8000
duration = 0.005  # 5 ms，足够看清楚
n_samples = int(sr * duration)

print("手工演算：采样点是如何被 sin() 函数计算的\n")
print(f"{'n':<3} | {'时间t(s)':<10} | {'6kHz圆周':<12} | {'sin(6kHz)':<10} | {'2kHz圆周':<12} | {'sin(2kHz)':<10} | {'-sin(2kHz)':<10}")
print("-" * 100)

for n in range(min(8, n_samples)):
    t = n / sr
    
    # 6 kHz 的相位和采样值
    phase_6k = 2 * np.pi * 6000 * t
    cycles_6k = 6000 * t
    cycles_6k_wrapped = (cycles_6k % 1)  # 只看小数部分
    sin_6k = np.sin(phase_6k)
    
    # 2 kHz 的相位和采样值
    phase_2k = 2 * np.pi * 2000 * t
    cycles_2k = 2000 * t
    sin_2k = np.sin(phase_2k)
    
    # 打印结果
    print(f"{n:<3} | {t:<10.6f} | {cycles_6k:<12.4f} | {sin_6k:<10.6f} | {cycles_2k:<12.4f} | {sin_2k:<10.6f} | {-sin_2k:<10.6f}")

print("\n观察：")
print("- '6kHz圆周' 列显示 6kHz 波转过多少圈（小数表示圈的一部分）")
print("- '2kHz圆周' 列显示 2kHz 波转过多少圈")
print("- 关键观察：6kHz 的采样值 = 2kHz 的采样值取反！（右边两列逐点相反）")
print("- 这就是混叠：6kHz 的采样点序列完全冒充了 2kHz 的反相版本。")

## 🎨 图示：混叠——6kHz 被 8kHz 采样后伪装成低频

In [ ]:
import aurora.aviz as aviz; aviz.style()
aviz.aliasing(6000, 8000);

In [ ]:
sr = 12
nyq = sr / 2
for f in [1, 5, 6, 7, 11, 13, 17]:
    alias = abs(((f + nyq) % sr) - nyq)
    status = '安全' if f <= nyq else '伪装'
    print(f'{f:>2}Hz -> {alias:>4.1f}Hz  {status}')


## 参数实验：扫描 0–32000 Hz，观察折叠规律

固定 `sr=16000`，把 `freq` 从 0 扫到 32000，每 1000 Hz 打印 `predict_alias_freq`，观察结果在 0–8000 Hz 范围内折叠：16000 Hz 折回 0 Hz，24000 Hz 折回 8000 Hz，32000 Hz 再次折回 0 Hz。把下面代码粘贴到空白单元格运行：

```python
sr = 16000
for f in range(0, 33000, 1000):
    alias = predict_alias_freq(f, sr)
    print(f'{f:>6} Hz  ->  {alias:>6.0f} Hz')
```

规律：折叠图案每隔 `sr` Hz 重复一次；Nyquist（8000 Hz）是对称轴，9000 Hz 和 7000 Hz 产生相同的视在频率（都是 1000 Hz）。

## 参数实验前：理解折叠的周期性和两个公式

### 为什么折叠图案每隔 sr 重复？

假设 alias(f, sr) 是折叠公式。我们要证明：**alias(f + sr, sr) = alias(f, sr)**

**推导**：

$$\text{alias}(f + sr) = |f + sr - sr \cdot \text{round}\!\left(\frac{f + sr}{sr}\right)|$$

$$= |f + sr - sr \cdot \left(\text{round}\!\left(\frac{f}{sr}\right) + 1\right)|$$

$$= |f + sr - sr \cdot \text{round}\!\left(\frac{f}{sr}\right) - sr|$$

$$= |f - sr \cdot \text{round}\!\left(\frac{f}{sr}\right)|$$

$$= \text{alias}(f)$$

**结论**：折叠图案以 sr 为周期重复。这就是为什么 0–sr 范围内的所有频率已经覆盖了全频域的信息。

### 两个折叠公式的等价性

在代码中，你可能看到过两种写法：

**公式 1**（标准形式）：
$$\text{alias}_1(f) = \left| f - sr \cdot \text{round}\!\left(\frac{f}{sr}\right) \right|$$

**公式 2**（模运算形式）：
$$\text{alias}_2(f) = \left| \left(\left(f + \text{nyquist}\right) \bmod sr\right) - \text{nyquist} \right|$$

其中 nyquist = sr/2。

**它们为什么等价？**

公式 2 的核心思想是：
- 先把频率"平移"到 nyquist（加上 nyquist）
- 用模运算把它折回 0–sr 范围（%sr）
- 再"反平移"回去（减去 nyquist）
- 结果就落在 -nyquist 到 +nyquist（也就是 -sr/2 到 +sr/2）范围内

这两个公式在数学上等价，但公式 1 更直观（直接找最近的折叠中心），公式 2 在处理周期时更方便。**本课使用公式 1**。

---

## 本课收束

现在可以用 `predict_alias_freq(freq, sr)` 把任意频率映射到 0–Nyquist 范围内的视在频率。画图验证会显示：6 kHz 高频的采样点落在 2 kHz 低频的反相曲线上（正弦向下折叠附带相位翻转），在离散域无法区分频率。Aurora 音频管道在 16 kHz 采样率下只提取 0–8 kHz 的 mel 特征；8 kHz 以上的能量如果没被抗混叠滤波器截断，就会变成假的低频，污染模型输入。下一节（L35，欧拉公式与旋转因子）引入复数表示，为后面推导 DFT 打基础；L36 再引入汉宁窗压制频谱泄漏。

In [ ]:
# 小检查：从短序列开始，确认每一步输出
import numpy as np

sample_rate = 8
duration = 1.0
freq = 1.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
x = np.sin(angle)

print('1) N 应该是多少？', N)
print('2) n 是采样点编号：', n)
print('3) t 是每个点的秒数：', np.round(t, 3))
print('4) angle 是每个点在圆上的角度：', np.round(angle, 3))
print('5) x 是最终波形：', np.round(x, 3))


---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：混叠手算（目标 10 分钟）

盖上屏幕，纸上作答。提示：用公式 alias = |f - sr·round(f/sr)|，每一步都要明确地写出 round() 的中间值。

**问 1**：sr=8000 Hz，f=6000 Hz
- 第一步：计算 f/sr = 6000/8000 = ?
- 第二步：round(?) = ?（是舍到哪个整数？）
- 第三步：sr × round(...) = ?
- 第四步：|6000 - ?| = alias

**问 2**：sr=8000 Hz，f=3000 Hz（低于 Nyquist=4000）
- 这个频率在 Nyquist 以下，应该**不折叠**，返回原值
- 用公式验证：round(3000/8000) = ?, 然后算

**问 3**：sr=8000 Hz，f=9000 Hz
- 这次 f > sr，折叠超过一圈
- 第一步：f/sr = 9000/8000 = 1.125，round(1.125) = ?
- 继续算下去

**问 4**：为什么 CD（44100 Hz）能录 20 kHz 人声上限？
- Nyquist 条件：sr > 2·f_max，即 sr 需要超过多少？
- 44100/2 = ? Hz（是否超过 20000？）
- 这才是 CD 采样率的由来

**问 5**：Aurora 音频管道：如果模型输入信号上限 8 kHz，采样率最低需要多少？
- 从 Nyquist 条件反推

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：sr=8000, f=6000 → alias=2000
sr = 8000
assert np.isclose(abs(6000 - sr * round(6000/sr)), 2000.0)
try:
    a1 = predict_alias_freq(6000, sr)
    assert np.isclose(a1, 2000.0), f"alias(6000,8000) 应=2000，得到 {a1}"
    print(f"Q1 ✅  alias(6000, 8000) = {a1} Hz（6kHz折叠为2kHz）")
except (NotImplementedError, TypeError):
    print("⬜ Q1：请先实现 predict_alias_freq()，再运行对答案格")

# 问2：sr=8000, f=3000 → alias=3000（不折叠）
assert np.isclose(abs(3000 - sr * round(3000/sr)), 3000.0)
try:
    a2 = predict_alias_freq(3000, sr)
    assert np.isclose(a2, 3000.0)
    print(f"Q2 ✅  alias(3000, 8000) = {a2} Hz（低于Nyquist=4000，不折叠）")
except (NotImplementedError, TypeError):
    print("⬜ Q2：请先实现 predict_alias_freq()，再运行对答案格")

# 问3：sr=8000, f=9000 → alias=1000
assert np.isclose(abs(9000 - sr * round(9000/sr)), 1000.0)
try:
    a3 = predict_alias_freq(9000, sr)
    assert np.isclose(a3, 1000.0)
    print(f"Q3 ✅  alias(9000, 8000) = {a3} Hz（round(9000/8000)=1，|9000-8000|=1000）")
except (NotImplementedError, TypeError):
    print("⬜ Q3：请先实现 predict_alias_freq()，再运行对答案格")

# 问4：CD 44100 Hz
nyquist_cd = 44100 / 2
assert nyquist_cd == 22050.0 and nyquist_cd > 20000
print(f"Q4 ✅  44100/2 = {nyquist_cd} Hz > 20000 Hz（人耳上限），Nyquist 满足")

# 问5：Aurora 8kHz 上限 → sr > 16000
min_sr = 2 * 8000
print(f"Q5 ✅  最低采样率 = 2×8000 = {min_sr} Hz，Aurora 常用 16000 Hz 正好满足")
print("\n🎉 混叠白板挑战通过！折叠公式 |f - sr·round(f/sr)| 已内化。")

### 深入理解：为什么 Nyquist 频率（sr/2）本身不折叠？

有学生问："混叠是高频折叠成低频的现象，那 4000 Hz 为什么不产生混叠，还是返回 4000？"

**答案**：4000 Hz 恰好是 8000 Hz 采样率的极限频率，它**既不折叠，也不算超过极限**。

数学上：
- f = 4000 Hz，sr = 8000 Hz
- f/sr = 0.5，round(0.5) = 0（Python 银行家舍入）
- fold center = 0 × 8000 = 0 Hz
- alias = |4000 - 0| = 4000 Hz

所以 4000 Hz 返回自己。

**为什么概念上也讲得通**：
- 4 kHz 波在每个采样间隔内转 0.5 圈（恰好 180°）
- 每两个采样点，波完成一个完整周期
- 这是能分辨的最高频率——你能看到振幅在"最大-最小-最大"交替
- 频率再高就看不清了（混叠），但 4 kHz 本身不超过这个极限

---

In [ ]:
# ✏️ 本课自评
l34_review = {
    "nyquist_theorem":           None,  # 记住 sr > 2f_max 的奈奎斯特条件？True/False
    "alias_formula":             None,  # 记住 alias=|f - sr·round(f/sr)|？True/False
    "predict_alias_implemented": None,  # predict_alias_freq 实现并通过断言？True/False
    "cd_sampling_rate":          None,  # 理解44100Hz为何能覆盖20kHz人耳上限？True/False
    "whiteboard_passed":         None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l34_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l34_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L34 全部通关！进入 L35：欧拉公式遇见 FFT')

---

→ **下一课**　[L35 · 欧拉公式遇见 FFT](L35_euler_fft.ipynb)

> 下节课将学习 **欧拉公式遇见 FFT**：e^{-2πikn/N} 是什么，旋转因子可视化。